In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

random.seed(42)
np.random.seed(42)

agents = [
    "Priya Sharma", "Rahul Verma", "Anita Singh", "Karan Mehta",
    "Sneha Gupta", "Vikram Nair", "Pooja Joshi", "Arjun Kapoor",
    "Neha Tiwari", "Rohit Das"
]

rows = []
start_date = datetime(2024, 1, 1)

for day in range(90):
    date = start_date + timedelta(days=day)
    if date.weekday() >= 5:  # skip weekends
        continue
    for agent in agents:
        claimed = random.randint(80, 120)
        variance = random.choice([-20, -10, -5, 0, 0, 0, 5, 10])
        actual = max(0, claimed + variance)
        hour = random.choices(
            [15, 16, 17, 18, 19, 20],
            weights=[30, 30, 20, 10, 7, 3]
        )[0]
        minute = random.randint(0, 59)
        submitted_at = date.replace(hour=hour, minute=minute)
        rows.append({
            "date": date.strftime("%Y-%m-%d"),
            "agent_name": agent,
            "tasks_claimed": claimed,
            "tasks_actual": actual,
            "submitted_at": submitted_at.strftime("%Y-%m-%d %H:%M"),
            "team": random.choice(["Team A", "Team B"])
        })

df = pd.DataFrame(rows)
df.to_csv("compliance_tracker.csv", index=False)
print(df.head(10))
print(f"\nTotal rows: {len(df)}")

         date    agent_name  tasks_claimed  tasks_actual      submitted_at  \
0  2024-01-01  Priya Sharma            120           110  2024-01-01 15:17   
1  2024-01-01   Rahul Verma             94            89  2024-01-01 17:43   
2  2024-01-01   Anita Singh            117           122  2024-01-01 15:05   
3  2024-01-01   Karan Mehta             94            74  2024-01-01 16:45   
4  2024-01-01   Sneha Gupta             94           104  2024-01-01 16:51   
5  2024-01-01   Vikram Nair             90            95  2024-01-01 16:09   
6  2024-01-01   Pooja Joshi            101            91  2024-01-01 15:06   
7  2024-01-01  Arjun Kapoor            102           102  2024-01-01 18:46   
8  2024-01-01   Neha Tiwari            114           104  2024-01-01 20:24   
9  2024-01-01     Rohit Das            115           115  2024-01-01 18:39   

     team  
0  Team A  
1  Team A  
2  Team A  
3  Team B  
4  Team A  
5  Team A  
6  Team B  
7  Team B  
8  Team A  
9  Team B  

Total ro

In [3]:
import subprocess
subprocess.run(["pip", "install", "psycopg2-binary", "sqlalchemy"])

CompletedProcess(args=['pip', 'install', 'psycopg2-binary', 'sqlalchemy'], returncode=0)

In [11]:
from sqlalchemy import create_engine

engine = create_engine(
    'postgresql+psycopg2://postgres@localhost:5432/compliance_tracker',
    connect_args={"password": "admin123"}
)

df.to_sql('compliance_tracker', engine, if_exists='replace', index=False)
print("Table created successfully!")

Table created successfully!


In [12]:
import pandas as pd

query1 = """
SELECT 
    agent_name,
    ROUND(AVG(tasks_claimed)) AS avg_claimed,
    ROUND(AVG(tasks_actual)) AS avg_actual,
    ROUND(AVG(tasks_claimed - tasks_actual)) AS avg_gap
FROM compliance_tracker
GROUP BY agent_name
ORDER BY avg_gap DESC;
"""

df_gap = pd.read_sql(query1, engine)
print(df_gap)

     agent_name  avg_claimed  avg_actual  avg_gap
0     Rohit Das         99.0        95.0      4.0
1   Anita Singh         98.0        94.0      4.0
2   Karan Mehta        100.0        97.0      3.0
3  Arjun Kapoor        101.0        99.0      2.0
4   Rahul Verma        100.0        98.0      2.0
5   Pooja Joshi        100.0        98.0      2.0
6   Neha Tiwari         99.0        96.0      2.0
7  Priya Sharma        102.0       100.0      2.0
8   Vikram Nair         99.0        98.0      1.0
9   Sneha Gupta        100.0       100.0      0.0


In [13]:
query2 = """
SELECT 
    agent_name,
    COUNT(*) AS total_days,
    SUM(CASE WHEN EXTRACT(HOUR FROM submitted_at::timestamp) >= 18 
        THEN 1 ELSE 0 END) AS late_submissions,
    ROUND(100.0 * SUM(CASE WHEN EXTRACT(HOUR FROM submitted_at::timestamp) >= 18 
        THEN 1 ELSE 0 END) / COUNT(*), 1) AS late_pct
FROM compliance_tracker
GROUP BY agent_name
ORDER BY late_pct DESC;
"""

df_late = pd.read_sql(query2, engine)
print(df_late)

     agent_name  total_days  late_submissions  late_pct
0   Karan Mehta          65                20      30.8
1   Sneha Gupta          65                18      27.7
2  Arjun Kapoor          65                17      26.2
3   Rahul Verma          65                17      26.2
4   Pooja Joshi          65                13      20.0
5  Priya Sharma          65                13      20.0
6   Neha Tiwari          65                13      20.0
7   Vikram Nair          65                11      16.9
8     Rohit Das          65                10      15.4
9   Anita Singh          65                 9      13.8


In [14]:
query3 = """
SELECT 
    team,
    ROUND(AVG(tasks_claimed)) AS avg_claimed,
    ROUND(AVG(tasks_actual)) AS avg_actual,
    ROUND(AVG(tasks_claimed - tasks_actual)) AS avg_gap,
    ROUND(100.0 * SUM(CASE WHEN EXTRACT(HOUR FROM submitted_at::timestamp) >= 18 
        THEN 1 ELSE 0 END) / COUNT(*), 1) AS late_pct
FROM compliance_tracker
GROUP BY team
ORDER BY team;
"""

df_team = pd.read_sql(query3, engine)
print(df_team)

     team  avg_claimed  avg_actual  avg_gap  late_pct
0  Team A         99.0        97.0      2.0      22.0
1  Team B        101.0        98.0      2.0      21.4


In [15]:
query4 = """
SELECT 
    agent_name,
    team,
    ROUND(AVG(tasks_actual)) AS avg_actual,
    ROUND(AVG(tasks_claimed - tasks_actual)) AS avg_gap,
    ROUND(100.0 * SUM(CASE WHEN EXTRACT(HOUR FROM submitted_at::timestamp) >= 18 
        THEN 1 ELSE 0 END) / COUNT(*), 1) AS late_pct,
    CASE 
        WHEN AVG(tasks_claimed - tasks_actual) <= 1 
        AND SUM(CASE WHEN EXTRACT(HOUR FROM submitted_at::timestamp) >= 18 
            THEN 1 ELSE 0 END) * 100.0 / COUNT(*) <= 20
        THEN 'Green'
        WHEN AVG(tasks_claimed - tasks_actual) <= 3 
        AND SUM(CASE WHEN EXTRACT(HOUR FROM submitted_at::timestamp) >= 18 
            THEN 1 ELSE 0 END) * 100.0 / COUNT(*) <= 25
        THEN 'Amber'
        ELSE 'Red'
    END AS compliance_status
FROM compliance_tracker
GROUP BY agent_name, team
ORDER BY compliance_status, avg_gap DESC;
"""

df_compliance = pd.read_sql(query4, engine)
print(df_compliance)

      agent_name    team  avg_actual  avg_gap  late_pct compliance_status
0    Neha Tiwari  Team B        95.0      3.0      18.4             Amber
1    Pooja Joshi  Team B       102.0      2.0      25.0             Amber
2    Neha Tiwari  Team A        98.0      2.0      22.2             Amber
3    Rahul Verma  Team B        97.0      2.0      22.6             Amber
4      Rohit Das  Team A        95.0      2.0      11.1             Amber
5    Pooja Joshi  Team A        95.0      1.0      15.2             Amber
6   Priya Sharma  Team B       101.0      0.0      21.6             Amber
7    Vikram Nair  Team B        99.0      0.0      23.1             Amber
8    Sneha Gupta  Team A        98.0      0.0      21.6             Amber
9   Arjun Kapoor  Team B       103.0      0.0      16.7             Green
10  Priya Sharma  Team A        98.0      6.0      17.9               Red
11   Karan Mehta  Team B        96.0      5.0      23.8               Red
12     Rohit Das  Team B        95.0  

In [16]:
# Save all results to Excel — one sheet per query
with pd.ExcelWriter('compliance_analysis.xlsx') as writer:
    df_gap.to_excel(writer, sheet_name='Overclaiming Analysis', index=False)
    df_late.to_excel(writer, sheet_name='Late Submissions', index=False)
    df_team.to_excel(writer, sheet_name='Team Comparison', index=False)
    df_compliance.to_excel(writer, sheet_name='Compliance Scorecard', index=False)

print("Excel file saved!")

Excel file saved!
